# Pandas with the ETL pipelines

In [60]:
#Importing necessary Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [61]:
#Ignoring the warnings

import warnings
warnings.filterwarnings('ignore')

In [62]:
#Reading the excel file

df=pd.read_csv('/content/messy_sales_data.csv')

In [63]:
df.head()  #Top rows

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.00,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.00,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.00,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.00,45000,2024-01-05,Mumbai,Anil Sharma


In [64]:
df.isnull()  # Checking the null values

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,False,False,False,False,False,False,False,False,False
1,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False
4,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False
7,False,True,False,False,False,False,False,False,False
8,False,False,False,False,False,False,False,False,False
9,False,False,False,False,False,False,False,False,False


In [65]:
df.isnull().sum()  #Total null values by Columns

,0
order_id,0
customer_name,2
product,1
category,1
quantity,3
unit_price,0
order_date,0
city,0
sales_rep,0


## Data Cleaning

In [66]:
df['quantity'].fillna(df['quantity'].mean(),inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
df['customer_name'].fillna('Unkwown_Customer',inplace=True)
df['product'].fillna('Unknown_product',inplace=True)
mode_cat = df['category'].mode()[0]
df['category'].fillna(mode_cat, inplace=True)

In [67]:
df.isnull().sum()

,0
order_id,0
customer_name,0
product,0
category,0
quantity,0
unit_price,0
order_date,0
city,0
sales_rep,0


In [68]:
print(f"Before Duplication : {len(df)} rows")
print(f"Duplicate rows: {df.duplicated().sum()}")






Before Duplication : 30 rows
Duplicate rows: 0


In [69]:
# The dates are in different format so we need a specified format :

print("Sample Dates before parsing :")
print(df['order_date'].head(10).to_list())

df['order_date']=pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')

df['order_year']=df['order_date'].dt.year
df['order_month']=df['order_date'].dt.month
df['order_day']=df['order_date'].dt.day

# The line below caused data corruption by adding numeric date components to all columns.
# It has been removed to preserve the integrity of other columns like 'customer_name'.
# df=df+df[['order_day','order_month','order_year']]

df.drop(['order_date'],inplace=True,axis=1)



Sample Dates before parsing :
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']


In [70]:
df.head()

,order_id,customer_name,product,category,quantity,unit_price,city,sales_rep,order_year,order_month,order_day
0,1001,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024.00,1.00,5.00
1,1002,Priya Nair,Unknown_product,Electronics,1.00,15000,Delhi,Sunita Rao,2024.00,1.00,7.00
2,1003,AMIT VERMA,Keyboard,Accessories,3.00,1200,Bangalore,Anil Sharma,2024.00,1.00,8.00
3,1004,Sunita Patel,Monitor,Electronics,3.93,22000,Chennai,Ravi Kumar,2024.00,1.00,10.00
4,1005,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024.00,1.00,5.00


In [71]:
#Converting the dates into nullable integer types to handle potential NaN values :

df['order_year']=df['order_year'].astype(pd.Int64Dtype())
df['order_month']=df['order_month'].astype(pd.Int64Dtype())
df['order_day']=df['order_day'].astype(pd.Int64Dtype())

In [72]:
df.head()

,order_id,customer_name,product,category,quantity,unit_price,city,sales_rep,order_year,order_month,order_day
0,1001,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024,1,5
1,1002,Priya Nair,Unknown_product,Electronics,1.00,15000,Delhi,Sunita Rao,2024,1,7
2,1003,AMIT VERMA,Keyboard,Accessories,3.00,1200,Bangalore,Anil Sharma,2024,1,8
3,1004,Sunita Patel,Monitor,Electronics,3.93,22000,Chennai,Ravi Kumar,2024,1,10
4,1005,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024,1,5


In [73]:
# Check product columns
print("Unique Products :")
print(df['product'].unique())

# Check category columns
print("Unique Categories :")
print(df['category'].unique())

Unique Products :
['Laptop' 'Unknown_product' 'Keyboard' 'Monitor' 'Mouse' 'Headphones'
 'Webcam' 'USB Hub']
Unique Categories :
['Electronics' 'Accessories']


In [74]:
# fix that keyboard as in electronics in category

df.loc[df['product'] == 'Keyboard', 'category'] = 'Electronics'

print(df[['order_id','product','category']])

    order_id          product     category
0       1001           Laptop  Electronics
1       1002  Unknown_product  Electronics
2       1003         Keyboard  Electronics
3       1004          Monitor  Electronics
4       1005           Laptop  Electronics
5       1006            Mouse  Accessories
6       1007       Headphones  Electronics
7       1008           Webcam  Accessories
8       1009           Laptop  Electronics
9       1010         Keyboard  Electronics
10      1011          Monitor  Electronics
11      1012          USB Hub  Accessories
12      1013           Laptop  Electronics
13      1014       Headphones  Electronics
14      1015           Webcam  Accessories
15      1016            Mouse  Accessories
16      1017         Keyboard  Electronics
17      1018          Monitor  Electronics
18      1019           Laptop  Electronics
19      1020          USB Hub  Accessories
20      1021       Headphones  Electronics
21      1022           Webcam  Accessories
22      102

In [75]:
# Text Standardization :

# Standardize customer names to title case
print("Before Standardization :", df['customer_name'].unique()[:6])

df['customer_name'] = (
    df['customer_name'].str.strip().str.title()
)

# After Standardization
print("After Standardization :", df['customer_name'].unique()[:6])

print("Cleaned rows:")
print(df.head())


Before Standardization : ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After Standardization : ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']
Cleaned rows:
   order_id customer_name          product     category  quantity  unit_price  \
0      1001  Ramesh Kumar           Laptop  Electronics      2.00       45000   
1      1002    Priya Nair  Unknown_product  Electronics      1.00       15000   
2      1003    Amit Verma         Keyboard  Electronics      3.00        1200   
3      1004  Sunita Patel          Monitor  Electronics      3.93       22000   
4      1005  Ramesh Kumar           Laptop  Electronics      2.00       45000   

        city    sales_rep  order_year  order_month  order_day  
0     Mumbai  Anil Sharma        2024            1          5  
1      Delhi   Sunita Rao        2024            1          7  
2  Bangalore  Anil Sharma        2024            1          8  
3    Chennai   Rav

In [76]:
import pandas as pd

# Ensure numeric conversion
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# Create revenue column
df['revenue'] = df['quantity'] * df['unit_price']
print("Revenue column created:\n")
print(df.head(5))

# Round revenue to 2 decimals
df['revenue'] = df['revenue'].round(2)

pd.options.display.float_format = '{:.2f}'.format

print("===============================================\n")
print("Total revenue across all rows : ₹", df['revenue'].sum())


Revenue column created:

   order_id customer_name          product     category  quantity  unit_price  \
0      1001  Ramesh Kumar           Laptop  Electronics      2.00       45000   
1      1002    Priya Nair  Unknown_product  Electronics      1.00       15000   
2      1003    Amit Verma         Keyboard  Electronics      3.00        1200   
3      1004  Sunita Patel          Monitor  Electronics      3.93       22000   
4      1005  Ramesh Kumar           Laptop  Electronics      2.00       45000   

        city    sales_rep  order_year  order_month  order_day  revenue  
0     Mumbai  Anil Sharma        2024            1          5 90000.00  
1      Delhi   Sunita Rao        2024            1          7 15000.00  
2  Bangalore  Anil Sharma        2024            1          8  3600.00  
3    Chennai   Ravi Kumar        2024            1         10 86370.37  
4     Mumbai  Anil Sharma        2024            1          5 90000.00  

Total revenue across all rows : ₹ 951851.85000000

In [77]:
len(df)

30

In [78]:
# Making as a report

print("=" *55)
print(' Post Cleaning validation Report ')
print("="*55)
print(f"Original rows : {len(df)}")
print(f"Missing Values : {df.isnull().sum().sum()}")
print(f"Duplicate Rows : {df.duplicated().sum()}")
print(f"Revenue Nan    : {df['revenue'].isnull().sum()}")
print(f"Categories    : {df['category'].unique()}")
print(f"Products      : {df['product'].unique()}")
print(f"Total Revenue : ₹{df['revenue'].sum()}")
print("="*55)





 Post Cleaning validation Report 
Original rows : 30
Missing Values : 6
Duplicate Rows : 0
Revenue Nan    : 0
Categories    : ['Electronics' 'Accessories']
Products      : ['Laptop' 'Unknown_product' 'Keyboard' 'Monitor' 'Mouse' 'Headphones'
 'Webcam' 'USB Hub']
Total Revenue : ₹951851.8500000001


In [79]:
df.isnull().sum()

,0
order_id,0
customer_name,0
product,0
category,0
quantity,0
unit_price,0
city,0
sales_rep,0
order_year,2
order_month,2


In [80]:
df

,order_id,customer_name,product,category,quantity,unit_price,city,sales_rep,order_year,order_month,order_day,revenue
0,1001,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024,1,5,90000.00
1,1002,Priya Nair,Unknown_product,Electronics,1.00,15000,Delhi,Sunita Rao,2024,1,7,15000.00
2,1003,Amit Verma,Keyboard,Electronics,3.00,1200,Bangalore,Anil Sharma,2024,1,8,3600.00
3,1004,Sunita Patel,Monitor,Electronics,3.93,22000,Chennai,Ravi Kumar,2024,1,10,86370.37
4,1005,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024,1,5,90000.00
5,1006,Kiran Mehta,Mouse,Accessories,10.00,800,Pune,Sunita Rao,<NA>,<NA>,<NA>,8000.00
6,1007,Deepak Singh,Headphones,Electronics,2.00,3500,Hyderabad,Ravi Kumar,2024,1,12,7000.00
7,1008,Unkwown_Customer,Webcam,Accessories,1.00,2500,Mumbai,Anil Sharma,2024,1,13,2500.00
8,1009,Ananya Das,Laptop,Electronics,1.00,45000,Kolkata,Sunita Rao,2024,1,15,45000.00
9,1010,Vikram Iyer,Keyboard,Electronics,5.00,1200,Chennai,Ravi Kumar,2024,1,15,6000.00


In [81]:
df[['order_day','order_month','order_year']]=df[['order_day','order_month','order_year']].fillna({'order_day': 15, 'order_month': 1, 'order_year': 2024})

In [82]:
df.isnull().sum()

,0
order_id,0
customer_name,0
product,0
category,0
quantity,0
unit_price,0
city,0
sales_rep,0
order_year,0
order_month,0


In [83]:

product_rev= (
    df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values(by='revenue',ascending=False)
)

print("Revenue by Product :")
print(product_rev.to_string(index=False))
print("="*55)
print("\n\n\n")

# Categorical Summary

category_summary = df.groupby('category').agg(

   total_revenue=('revenue', 'sum'),
   total_customers=('customer_name','nunique'),
   avg_revenue_per_customer=('revenue','mean'),
   total_orders=('order_id','count'),
   avg_orders_per_customer=('order_id','mean'),
   avg_quantity_per_order=('quantity','mean'),
   avg_revenue_per_order=('revenue','mean')

)

print("\nCategory Summary :")
print(category_summary.to_string())

Revenue by Product :
        product   revenue
         Laptop 626666.67
        Monitor 196370.37
     Headphones  28000.00
         Webcam  24814.81
          Mouse  20800.00
       Keyboard  20400.00
        USB Hub  19800.00
Unknown_product  15000.00





Category Summary :
             total_revenue  total_customers  avg_revenue_per_customer  total_orders  avg_orders_per_customer  avg_quantity_per_order  avg_revenue_per_order
category                                                                                                                                                   
Accessories       65414.81                9                   6541.48            10                  1018.30                    6.89                6541.48
Electronics      886437.04               17                  44321.85            20                  1014.10                    2.44               44321.85


In [84]:
# Save as CSV :

df.to_csv("Cleaned_data")

In [85]:
New_df=pd.read_csv('/content/Cleaned_data')

New_df.head()

,Unnamed: 0,order_id,customer_name,product,category,quantity,unit_price,city,sales_rep,order_year,order_month,order_day,revenue
0,0,1001,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024,1,5,90000.00
1,1,1002,Priya Nair,Unknown_product,Electronics,1.00,15000,Delhi,Sunita Rao,2024,1,7,15000.00
2,2,1003,Amit Verma,Keyboard,Electronics,3.00,1200,Bangalore,Anil Sharma,2024,1,8,3600.00
3,3,1004,Sunita Patel,Monitor,Electronics,3.93,22000,Chennai,Ravi Kumar,2024,1,10,86370.37
4,4,1005,Ramesh Kumar,Laptop,Electronics,2.00,45000,Mumbai,Anil Sharma,2024,1,5,90000.00


# Working with the API's

In [86]:
import requests

API_KEY = "d79f3111fac016cb6c7d7f57d2d8e568"

BASE_URL = "https://api.openweathermap.org/data/2.5/weather?"

cities = ['Mumbai','Delhi','Bangalore','Chennai','Hyderabad','Kolkata','Pune','Jaipur']

print(f" Api configured for  {len(cities)} cities")
print(f" Cities : {cities}")

weather_data = []

print("\nCalling the API and performing operations...")
for city in cities:
    params = {
        'q': city,
        'appid': API_KEY,
        'units': 'metric' # For temperature in Celsius
    }
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        data = response.json()
        main_data = data['main']
        weather_description = data['weather'][0]['description']
        weather_data.append({
            'City': city,
            'Temperature (C)': main_data['temp'],
            'Humidity (%)': main_data['humidity'],
            'Pressure (hPa)': main_data['pressure'],
            'Weather Description': weather_description
        })
        print(f"Successfully fetched data for {city}")
    else:
        print(f"Failed to fetch data for {city}. Status code: {response.status_code}")

weather_df = pd.DataFrame(weather_data)

print("\nWeather Data DataFrame:")
print(weather_df.to_string(index=False))



 Api configured for  8 cities
 Cities : ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

Calling the API and performing operations...
Successfully fetched data for Mumbai
Successfully fetched data for Delhi
Successfully fetched data for Bangalore
Successfully fetched data for Chennai
Successfully fetched data for Hyderabad
Successfully fetched data for Kolkata
Successfully fetched data for Pune
Successfully fetched data for Jaipur

Weather Data DataFrame:
     City  Temperature (C)  Humidity (%)  Pressure (hPa) Weather Description
   Mumbai            32.99            58            1009                haze
    Delhi            44.05            10             999           clear sky
Bangalore            29.69            55            1010    scattered clouds
  Chennai            35.68            56            1005          few clouds
Hyderabad            31.23            48            1010       broken clouds
  Kolkata            31.97            66